# DNN v3 Export — Raw Candle History (B1)

**Architecture:** ContextConditionedTCN with 116 features:
- 11 raw snapshot features (temporal branch)
- 105 prior candle OHLCV: 21 candles × 5 features (context branch via FiLM)

The DNN learns its own patterns from raw price history instead of pre-computed indicators.

Single-snapshot training, `elapsed_pct <= 0.50` cutoff.

In [1]:
import json
import os
import sqlite3
import sys

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, "../..")
from polybot.adapters.dnn_models import ContextConditionedTCN

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_COLS = [
    "btc_price",
    "elapsed_pct",
    "market_volume",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
]
CANDLE_LOOKBACK = 21
CANDLE_FEATURES = ["open", "high", "low", "close", "volume"]
PRIOR_CANDLE_COLS = [f"candle_{i}_{f}" for i in range(CANDLE_LOOKBACK) for f in CANDLE_FEATURES]
ALL_COLS = RAW_COLS + PRIOR_CANDLE_COLS  # 116 features

ELAPSED_CUTOFF = 0.50
MODELS_DIR = "../../models"
DATA_DIR = "../../data"

print(f"Features: {len(RAW_COLS)} raw + {len(PRIOR_CANDLE_COLS)} prior OHLCV = {len(ALL_COLS)} total")

Features: 11 raw + 105 prior OHLCV = 116 total


## 1. Load Data + Prior Candle OHLCV

In [ ]:
# Load snapshots
rows = []
with open("../../data/latest_features.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

# Load candle OHLCV from DB
conn = sqlite3.connect("file:../../data/collection.db?mode=ro", uri=True)
candles_db = pd.read_sql("SELECT candle_id, open, high, low, close, volume FROM candles ORDER BY start_time", conn)
conn.close()

candle_order = candles_db["candle_id"].tolist()
candle_ohlcv = candles_db[CANDLE_FEATURES].values.astype(np.float32)
candle_id_to_idx = {cid: i for i, cid in enumerate(candle_order)}

# Attach prior candle OHLCV (21 × 5 = 105 features per snapshot)
valid_cids = set()
prior_arrays = {}
for cid in df["candle_id"].unique():
    idx = candle_id_to_idx.get(cid)
    if idx is not None and idx >= CANDLE_LOOKBACK:
        prior_arrays[cid] = candle_ohlcv[idx - CANDLE_LOOKBACK : idx].flatten()
        valid_cids.add(cid)

df = df[df["candle_id"].isin(valid_cids)].copy()
prior_matrix = np.array([prior_arrays[cid] for cid in df["candle_id"]], dtype=np.float32)
prior_sub = pd.DataFrame(prior_matrix, columns=PRIOR_CANDLE_COLS, index=df.index)
df = pd.concat([df, prior_sub], axis=1)
df[ALL_COLS] = df[ALL_COLS].fillna(0)

print(f"Snapshots: {len(df):,} | Candles: {df['candle_id'].nunique():,}")

## 2. Train ContextConditionedTCN (single-snapshot, 116 features)

In [3]:
# Mid-candle, last snapshot per candle (single-snapshot training)
df_mid = df[df["elapsed_pct"] <= ELAPSED_CUTOFF]
candle_df = df_mid.groupby("candle_id").last().reset_index()

candle_ids = candle_df["candle_id"].values
split_idx = int(len(candle_df) * 0.8)

X_all = candle_df[ALL_COLS].values.astype(np.float32)
y_all = candle_df["target"].values.astype(np.float32)

X_train_raw, X_test_raw = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

model = ContextConditionedTCN(raw_size=len(RAW_COLS), context_size=len(PRIOR_CANDLE_COLS))
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ContextConditionedTCN: {params:,} params ({len(ALL_COLS)} features)")

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.BCEWithLogitsLoss()
X_t = torch.tensor(X_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=256, shuffle=True)

sp = int(len(X_train) * 0.9)
X_es = torch.tensor(X_train[sp:], dtype=torch.float32)
y_es = torch.tensor(y_train[sp:], dtype=torch.float32).unsqueeze(1)

best_vl, best_state, no_improve = float("inf"), None, 0
for epoch in range(1, 51):
    model.train()
    for bx, by in loader:
        optimizer.zero_grad()
        loss_fn(model(bx), by).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    model.eval()
    with torch.no_grad():
        vl = loss_fn(model(X_es), y_es).item()
    marker = ""
    if vl < best_vl:
        best_vl = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
        marker = " *"
    else:
        no_improve += 1
    if epoch % 5 == 0 or marker:
        print(f"  Epoch {epoch:2d}: val_loss={vl:.4f}{marker}")
    if no_improve >= 8:
        print(f"  Early stop at epoch {epoch}")
        break
if best_state:
    model.load_state_dict(best_state)
model.eval()
print(f"\nBest val_loss: {best_vl:.4f}")

/var/folders/j0/1_rf1mc97yb170c54jgv7lsr0000gn/T/ipykernel_31591/3222303360.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  candle_df = df_mid.groupby("candle_id").last().reset_index()


ContextConditionedTCN: 99,393 params (116 features)


  Epoch  1: val_loss=0.5789 *


  Epoch  2: val_loss=0.5358 *


  Epoch  3: val_loss=0.5332 *


...

  Epoch 43: val_loss=0.4633 *


  Epoch 45: val_loss=0.4584 *


  Epoch 46: val_loss=0.4555 *


  Epoch 49: val_loss=0.4548 *


  Epoch 50: val_loss=0.4503 *

Best val_loss: 0.4503


## 3. Calibrate & Export

In [4]:
X_v = torch.tensor(X_test, dtype=torch.float32)
model.eval()
with torch.no_grad():
    probs_raw = torch.sigmoid(model(X_v)).numpy().flatten()

calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
calibrator.fit(probs_raw, y_test)
probs_cal = calibrator.predict(probs_raw)

acc = accuracy_score(y_test, (probs_cal >= 0.5).astype(int))
brier = brier_score_loss(y_test, probs_cal)
f1 = f1_score(y_test, (probs_cal >= 0.5).astype(int), zero_division=0)

print(f"Raw:        Brier={brier_score_loss(y_test, probs_raw):.4f}")
print(f"Calibrated: Brier={brier:.4f}, Acc={acc * 100:.1f}%, F1={f1 * 100:.1f}%")

# Export
model_path = os.path.join(MODELS_DIR, "dnn_v1.pt")
torch.save(model, model_path)
print(f"\nModel: {model_path} ({os.path.getsize(model_path) / 1024:.0f} KB)")

scaler_path = os.path.join(MODELS_DIR, "dnn_scaler_v1.joblib")
joblib.dump(scaler, scaler_path)

calibrator_path = os.path.join(MODELS_DIR, "dnn_calibrator_v1.joblib")
joblib.dump(calibrator, calibrator_path)

feature_cols_path = os.path.join(MODELS_DIR, "dnn_feature_cols_v1.joblib")
joblib.dump(ALL_COLS, feature_cols_path)
print(f"Feature cols: {len(ALL_COLS)} features")

feature_config = {
    "model": "dnn_v3",
    "features": ALL_COLS,
    "n_features": len(ALL_COLS),
    "raw_features": len(RAW_COLS),
    "context_features": len(PRIOR_CANDLE_COLS),
    "context_type": "raw_candle_ohlcv",
    "candle_lookback": CANDLE_LOOKBACK,
    "architecture": "ContextConditionedTCN",
    "temporal": False,
    "calibrated": True,
    "parameters": params,
    "elapsed_cutoff": ELAPSED_CUTOFF,
    "metrics": {"accuracy": round(acc, 4), "brier": round(brier, 4), "f1": round(f1, 4)},
}
with open(os.path.join(DATA_DIR, "optimal_features_dnn.json"), "w") as f:
    json.dump(feature_config, f, indent=2)
print("All artifacts exported")

Raw:        Brier=0.1985
Calibrated: Brier=0.1898, Acc=71.5%, F1=73.8%

Model: ../../models/dnn_v1.pt (418 KB)
Feature cols: 116 features
All artifacts exported


## 4. Verify with DnnPredictor

In [5]:
from polybot.adapters.dnn_predictor import DnnPredictor
from polybot.ports.predictor import Predictor

predictor = DnnPredictor(
    model_path=model_path,
    feature_cols_path=feature_cols_path,
    scaler_path=scaler_path,
    calibrator_path=calibrator_path,
    temporal=False,
)

assert isinstance(predictor, Predictor)

sample_row = {col: float(X_test_raw[0, i]) for i, col in enumerate(ALL_COLS)}
p = predictor.predict(sample_row)
assert 0.0 <= p <= 1.0

print(f"DnnPredictor v3 loaded! temporal=False, {len(ALL_COLS)} features")
print(f"Sample prediction: P(UP) = {p:.4f}")

DnnPredictor v3 loaded! temporal=False, 116 features
Sample prediction: P(UP) = 0.1702
